# Section 2 Assignment - Automatic Text Generation

# Task
The task is to generate a number of versions of a short article entitled "The Implications of Recent Developments in Artificial Intelligence".

The exercise here it use:

- a variety of the decoding methods
- models and
- parameters

introduced through the course material to do this, while comparing and assessing the outputs.


Ideally, an article looks like:

- it was written by a human and looks credible,
- with reference to concrete facts,
- without using excessive computing.

It needs to be
- rich and informative, yet concise and complete.

Excessive diversity will result in jibberish, while restricted diversity will lead to repetition and will suggest a non-human auther.

You will need to consider how best to set up the prompts and the effects of increasing processing loads.

Consider, for each text generation choice,
- what the "giveaways" are that may suggest that the text is not written by a human.

When you see a good performance, show the effect of degrading it.

(Subsequently in the module I will introduce you to tools and metrics that will help with this, but for now I want to motivate your judgement of how such an NLU task would be assessed)


# Workbook format
- models used - gpt2-medium, qwen for comparison
- prompts
- decoding methods
- qwen comparison
- degradation


### Setup / Shared code
Uses example workbook code to create functions that are used to:

1. Build a model
1. Generate text using this model

In [1]:
import textwrap
from pprint import pprint
import pandas as pd
from IPython.display import display

pd.set_option('display.max_colwidth', 1000)

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

def buildModel(model_name) :

  device = "cuda" if torch.cuda.is_available() else "cpu"


  # download + load tokenizer for the model
  tokenizer = AutoTokenizer.from_pretrained(model_name)

  # load the model , to predict text
  model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

  return  tokenizer,model, device

In [3]:
def generate_text(tokenizer,model,device, prompt,  **text_gen_kwargs):

    input_ids = tokenizer(prompt, return_tensors="pt")["input_ids"].to(device)

    # do_sample = False means a greedy search
    output = model.generate(input_ids, **text_gen_kwargs)

    generated_text = tokenizer.decode(output[0])

    return { "prompt": prompt,"generated_text": generated_text, **text_gen_kwargs}

In [4]:
# https://huggingface.co/openai-community/gpt2-medium
# "gpt2-xl"
model_name = "gpt2-medium"

tokenizer,model, device = buildModel(model_name)

config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-medium
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

## Prompt

In [5]:
input_text = "The following is a short article written for a technology magazine.  The Implications of Recent Developments in Artificial Intelligence"

## Decoding methods
Hugging face parameters - https://huggingface.co/docs/transformers/main_classes/text_generation

max_new_tokens = 200 for all decodings

### Greedy Generation
**Configuration 1:**

    "max_new_tokens" : 200, "do_sample":False, "repetition_penalty" :1

**Configuration 2**:

    "max_new_tokens" : 200, "do_sample":False, "repetition_penalty" :1.3

In the 2nd configuration, repetition_penality is set to 1.3 to discourage the repetition seen in the 1st message.

In [11]:

args = [
    {"max_new_tokens" : 200, "do_sample":False, "repetition_penalty" :1},
    {"max_new_tokens" : 200, "do_sample":False, "repetition_penalty" :1.3}
]

results = []
for config in args:
    result = generate_text(tokenizer,model, device,input_text, **config)
    results.append(result)

result_df = pd.DataFrame(results)
display (result_df)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


,prompt,generated_text,max_new_tokens,do_sample,repetition_penalty
0,The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence,"The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence for the Future of Work and the Economy. \n\nThe Future of Work and the Economy\n\nThe future of work is not a question of whether we will have jobs, but rather of how we will manage them. The future of work is not a question of whether we will have jobs, but rather of how we will manage them.\n\nThe future of work is not a question of whether we will have jobs, but rather of how we will manage them.\n\nThe future of work is not a question of whether we will have jobs, but rather of how we will manage them.\n\nThe future of work is not a question of whether we will have jobs, but rather of how we will manage them.\n\nThe future of work is not a question of whether we will have jobs, but rather of how we will manage them.\n\nThe future of work is not a question of whether we will have jobs, but rather of how we will manage them.\n",200,False,1.0
1,The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence,"The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence and Robotics\n\nIn the past few years, artificial intelligence has become increasingly sophisticated at understanding human behavior through machine learning algorithms that are able to learn from experience or observe patterns within data sets (e-learning). This means we can now use these techniques on our own devices without having any prior knowledge about how they work – which makes them ideal candidates as tools used to improve humans' abilities with their everyday lives. However this does not mean there will be no need if AI becomes more intelligent than us: it could even lead directly into an era where machines take over jobs such like driving cars by using advanced computer vision systems instead! In fact, one recent study suggests robots may soon replace many manual workers who currently do most tasks around factories today; however some e...",200,False,1.3


### Analysis of generated text

**Text generated using Configuration 1**
> The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence for the Future of Work and the Economy. \n\nThe Future of Work and the Economy\n\nThe future of work is not a question of whether we will have jobs, but rather of how we will manage them. The future of work is not a question of whether we will have jobs, but rather of how we will manage them.\n\nThe future of work is not a question of whether we will have jobs, but rather of how we will manage them.\n\nThe future of work is not a question of whether we will have jobs, but rather of how we will manage them.\n\nThe future of work is not a question of whether we will have jobs, but rather of how we will manage them.\n\nThe future of work is not a question of whether we will have jobs, but rather of how we will manage them.\n\nThe future of work is not a question of whether we will have jobs, but rather of how we will manage them.\n

**Text generated using Configuration 2**
> The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence and Robotics\n\nIn the past few years, artificial intelligence has become increasingly sophisticated at understanding human behavior through machine learning algorithms that are able to learn from experience or observe patterns within data sets (e-learning). This means we can now use these techniques on our own devices without having any prior knowledge about how they work – which makes them ideal candidates as tools used to improve humans' abilities with their everyday lives. However this does not mean there will be no need if AI becomes more intelligent than us: it could even lead directly into an era where machines take over jobs such like driving cars by using advanced computer vision systems instead! In fact, one recent study suggests robots may soon replace many manual workers who currently do most tasks around factories today; however some e...

**Observations**
1. The first text contained repeated text "The future of work is not a question whether we will have jobs...them."  The model, has identified this phrase/sentence as the most likely successor to the preceeding phrasing.
1. Both texts are generated with the same configuration exception the second text specifies a repetition_penality of 1.3 to discourage repetition. This produces a more coherent body of text.

**Giveaways**

1. The 1st text is full of repeating text
1. The 2nd text has phrasing:
    - "improve humans' abilities"
    - "computer vision systems instead!"

which does not look correct.

2. The 2nd text uses a colon rather than a semi colon in the phrase "than us: it", which I think is incorrect.

**Conclusion**

Configuration two produces a readable sentence, suggesting that the repetition_penalty can improve greedy decoding / the quality of the genereated text. However, some punctuation and phrasing in text 2 would suggest that it is computer generated.

### Beam Search

In [13]:
args = [
    {"max_new_tokens" : 200, "do_sample":False, "num_beams":2},
    {"max_new_tokens" : 200, "do_sample":False, "num_beams":5},
    {"max_new_tokens" : 200, "do_sample":False, "num_beams":5, "no_repeat_ngram_size":2}
]

results = []
for config in args:
    result = generate_text(tokenizer,model, device,input_text, **config)
    results.append(result)

result_df = pd.DataFrame(results)
display (result_df)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


,prompt,generated_text,max_new_tokens,do_sample,num_beams,no_repeat_ngram_size
0,The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence,The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence for the Future of Work. \n\nThe Implications of Recent Developments in Artificial Intelligence for the Future of Work.\n\nThe Implications of Recent Developments in Artificial Intelligence for the Future of Work.\n\nThe Implications of Recent Developments in Artificial Intelligence for the Future of Work.\n\nThe Implications of Recent Developments in Artificial Intelligence for the Future of Work.\n\nThe Implications of Recent Developments in Artificial Intelligence for the Future of Work.\n\nThe Implications of Recent Developments in Artificial Intelligence for the Future of Work.\n\nThe Implications of Recent Developments in Artificial Intelligence for the Future of Work.\n\nThe Implications of Recent Developments in Artificial Intelligence for the Future of Work.\n\nThe Implications of Recent Developments in Artificial Intelligence for the Futur...,200,False,2,NaN
1,The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence,The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence and Machine Learning\n\nThe following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence and Machine Learning\n\nThe following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence and Machine Learning\n\nThe following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence and Machine Learning\n\nThe following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence and Machine Learning\n\nThe following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence and Machine Learning\n\nThe following is a short article wri...,200,False,5,NaN
2,The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence,The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence and Machine Learning for the Future of the Internet of Things (IoT). The article is based on a talk given at the 2016 IEEE International Conference on Computer Vision and Pattern Recognition (ICCV2016). The article will be published in an upcoming issue of IEEE Transactions on Intelligent Robots and Systems (ITRS) and will also be available on the ITRS website at http://www.itrs.ieee.org/index.php?option=com_content&view=article&id=10.1109/ITRRS.2016.63979 . _______________________________________________ Sent through the Full Disclosure mailing list https://noreply.github.io/mailman/listinfo/fulldisclosure Web Archives & RSS: http:/ / www.prnewswire.com/news-releases/implications-of-recent-developments-in-artificial-intelligence-and-machine-learning-for-the-,200,False,5,2.0


### Temperature


In [ ]:

args = {"do_sample":True, "max_length":  300, "top_k":0}

temperatures = [0.5, 0.75, 1, 1.25, 1.5]

results = []

for temperature in temperatures:
    args["temperature"] = temperature

    result = generate_text(tokenizer,model, device,input_text, **args)

    results.append(result)


result_df = pd.DataFrame(results)
display (result_df)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generati

,prompt,generated_text,do_sample,max_length,top_k,temperature
0,The Implications of Recent Developments in Artificial Intelligence,"The Implications of Recent Developments in Artificial Intelligence,"" in Artificial Intelligence: A Modern Approach, S. M. Stern, ed. Springer-Verlag, pp. 471-482.\n\n[6] See, for example, the recent paper by the authors of this paper on the nature of the ""intelligence explosion"" in the field of machine learning.\n\n[7] See, for example, the recent paper by the authors of this paper on the nature of the ""intelligence explosion"" in the field of machine learning.\n\n[8] See, for example, the recent paper by the authors of this paper on the nature of the ""intelligence explosion"" in the field of machine learning.\n\n[9] See, for example, the recent paper by the authors of this paper on the nature of the ""intelligence explosion"" in the field of machine learning.\n\n[10] See, for example, the recent paper by the authors of this paper on the nature of the ""intelligence explosion"" in the field of machine learning.\n\n[11] See, for example, the recent paper by the authors of this paper on th...",True,300,0,0.50
1,The Implications of Recent Developments in Artificial Intelligence,"The Implications of Recent Developments in Artificial Intelligence for the Future of Human Health and Well-Being,"" whose results were posted at the journal Health Affairs on Wednesday.\n\nThe researchers examined the effect of self-driving cars in the United States over the next five years. The paper says that self-driving cars are ""likely to play an even more important role in health and well-being over the next several decades"" than they do today.\n\nIndeed, the military, police and emergency crews are already using self-driving cars to ensure the safety of their operations. And Google is building a fleet of cars that can drive themselves.\n\nNow, it appears that the federal government is about to take a significant step in this direction.\n\nThe National Highway Traffic Safety Administration (NHTSA) has approved a request to begin testing self-driving cars on public roads throughout the country. The car will be able to operate at up to 60 mph and will be able to detect pedestria...",True,300,0,0.75
2,The Implications of Recent Developments in Artificial Intelligence,"The Implications of Recent Developments in Artificial Intelligence for the National Security of the United States"" read Jim Holleran, two-star Army General and head of the Defense Advanced Research Projects Agency. Holleran was giving the title presentation at the RAND conference in California this week.\n\nNo major investor seemed to notice, but none, perhaps, better than Brigadier General Michael Dunlavey, former Deputy Assistant Secretary of Defense for Science, Mathematics, and Intelligence. In September 2003, as head of the Defense Warrior Development Program, an elite program that in the former Soviet Union provided limited military training to young men and women, Dunlavey oversaw the creation of the Advanced Research Projects Agency complex within Kirtland Air Force Base in New Mexico.\n\nAlong with funding his much-hyped Defense Enhancement Program, like that 175,000 women serve in the army he spent much of the first 10 years of the 21st century making the mind-numbing des...",True,300,0,1.00
3,The Implications of Recent Developments in Artificial Intelligence,"The Implications of Recent Developments in Artificial Intelligence."") An NPR entry notes that previous passages had detailed how macroeconomic outcomes could change as a consequence of smarter robots. Lt. Cmdr. Eleftheria Olgheard uses robots in white paper onboard commercial 16-story super cruise liner Tribal If top AI chips systems make the system less vulnerable financially and more hard to hack, why even bother the Indian Armor? But there is an army XII658, which NurGam considers, a steampunkgish Jan Mary recommended Marvel Cthulhu Royal Guard personality portfolio, through 1 füte

### Top k

In [6]:
args = [{"do_sample":True, "max_length":  300,"top_k":20},
        {"do_sample":True, "max_length":  300,"top_k":50},
        {"do_sample":True, "max_length":  300,"top_k":100}]

results = []
for config in args:
    result = generate_text(tokenizer,model, device,input_text, **config)
    results.append(result)

result_df = pd.DataFrame(results)
display (result_df)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


,prompt,generated_text,do_sample,max_length,top_k
0,The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence,"The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence .\nI was on the receiving end of some bad programming. I'd done quite a bit of programming myself before and I'd learned from those experiences that I could program pretty well and that there was no limit to what I could accomplish in my head. But this time, the software came out of the programmer's mouth and said: This program is bad. You're going to pay a premium for it.\nThis is probably the most ridiculous piece of software out there, so I decided I should do my best to understand why this program was so bad.\nThis article is based on my experiences in programming using programming languages. It was written to help others understand how programming works, what the various programs are doing and, most importantly, how it can be applied to computer vision.\nIn the following section, I'll explain what a programming language is, how it works, some ...",True,300,20
1,The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence,"The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence. Artificial Intelligence has been around for decades. Today there is no denying that artificial intelligence can play a vital role in creating new tools and capabilities, not only in creating jobs and developing new platforms, but also in helping provide entertainment, commerce and communication technology. The technology is currently gaining ground at a rapid rate and is expected to be a catalyst for change in many fields. The technology is far from a done deal, and this article will try to outline many of the major issues that artificial intelligence will have on society sooner than later.\n\nThe Future of Jobs and Technology It's time to think about the long term. We need to understand the ramifications that artificial intelligence will have for the economy. Artificial intelligence is capable of doing things that computers cannot. In fact, AI al...",True,300,50
2,The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence,"The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence for the National Defense. Although the technology article is intended for general purposes and for a discussion of some of the aspects of Artificial Intelligence the effects are also applicable for national defense purposes. It is also listed as a list of the more important articles. For a more detailed account, see:\nposted 23.10.12 at 19:55\nHow Much is Enough by Michael R. Brown III/I.R.R. Davis This column is intended for the general reader who does not need precise information about the subject matter. It talks primarily about the application of Artificial Intelligence in the natural world; however, some of the concerns are realizations of the things we have been talking about. Other articles dealing with the implications of artificial intelligence and the National Defense are given first. References Michael Brown. "" How Much is Enough ."" Int...",True,300,100


Top p

In [7]:
args = [
    {"do_sample":True, "max_length":  300,"top_p":0.5},
    {"do_sample":True, "max_length":  300,"top_p":0.7},
    {"do_sample":True, "max_length":  300,"top_p":0.9}]

results = []
for config in args:
    result = generate_text(tokenizer,model, device,input_text, **config)
    results.append(result)

result_df = pd.DataFrame(results)
display (result_df)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


,prompt,generated_text,do_sample,max_length,top_p
0,The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence,"The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence for the Future of the World's Economy. The article was written by David R. Miller, Ph.D. and was published in the November 2012 issue of The World Economy. \n\nA new wave of artificial intelligence is now sweeping the world, and its impact is likely to be far-reaching. \n\nIn the next few years, artificial intelligence will be able to perform many tasks previously considered impossible, such as driving cars, finding lost people, and finding missing people. \n\nThis article will explore some of the implications of this development, and then provide some recommendations for how to prepare for the future. \n\nThe Implications of Recent Developments in Artificial Intelligence for the Future of the World's Economy\n\nA new wave of artificial intelligence is now sweeping the world, and its impact is likely to be far-reaching. \n\nIn the next few years, ...",True,300,0.5
1,The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence,"The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence. By Tom Williams. February 2004. This article is intended to be a reference and overview of some of the major developments in AI. It will be of interest to all those interested in AI, including those working in the field. _________________________________________________________ _________ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ _____ ____...",True,300,0.7
2,The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence,"The following is a short article written for a technology magazine. The Implications of Recent Developments in Artificial Intelligence and the Role of Natural Language Processing and Artificial Neural Networks for the Design of Computer Generated Artificial Intelligence, was published on July 27, 2017. It is an edited translation of an article published in Neural Computation (2017). The article contains a brief outline of the topic and is not comprehensive in details. The following are excerpts taken from the article that are added by way of context from the article and/or the corresponding book.\n\nThe Human Brain has a tremendous amount of power when compared to our bodies. This is because its capacity is greater than the limits of the human body. Therefore, there is a big potential for our brain to expand and develop. The main objective of artificial intelligence research is to increase this capacity, and artificial intelligence is already starting to take that step. As mention...",True,300,0.9


## Tools for the task
* Some great working examples can be found in the file **05_text_generation.ipynb** from the [Natural Language Processing with Transformers GitHub Repository](https://github.com/nlp-with-transformers/notebooks).
* There is a variety of models on Hugging Face that allow you to try out different models for your task. It is strongly recommended that you review these and look at the model cards in each case. Also, you can test your text in the Hosted inference API on the right hand side of each model.
* If you want to consider execution time of text generation with different decoding configurations or prompts, you can consider using [Python timing functions](https://realpython.com/python-timer/)

# Submission of your Assignment
Submit your main Jupyter notebook file and your articles for your assignment through Brightspace in the same manner as you have before. Near the top of the file, provide a link to your GitHub repository for the task which contains your input and output files.